# PHẦN II: PHÂN TÍCH VÀ BIẾN ĐỔI DỮ LIỆU CHUYÊN SÂU

Dataset: **Students Performance in Exams** (Kaggle) - học sinh với 5 cột phân loại và 3 cột điểm (math, reading, writing).

Chạy file này **sau Phần I**: đọc `students_cleaned.csv` (đã loại dòng trùng/thiếu và đổi tên cột về dạng `snake_case`), cuối file lưu `students_part2.csv` cho Phần III.

In [1]:
# Import pandas để làm việc với dữ liệu dạng bảng (DataFrame)
import pandas as pd
# Import numpy để tạo cột theo điều kiện (np.where)
import numpy as np
# Import os để kiểm tra thư mục shared có tồn tại hay không
import os

# ---------- settings (giống Phần I) ----------
# Đọc file đã làm sạch do Phần I lưu: ưu tiên thư mục shared, nếu không có thì dùng thư mục hiện tại
DUONG_DAN = "../shared/students_cleaned.csv" if os.path.exists("../shared/students_cleaned.csv") else "students_cleaned.csv"
TARGET = "math_score"          # cột điểm cần phân tích
GROUP_COL = "race_ethnicity"   # cột nhóm chính

# Đọc dữ liệu đã làm sạch (Phần I đã kiểm tra thiếu/trùng và đổi tên cột nên không làm lại ở đây)
df = pd.read_csv(DUONG_DAN)
print("Đã đọc:", DUONG_DAN, "| kích thước:", df.shape)
print("Các cột:", df.columns.tolist())

# Các cột điểm: lấy từ cột số, tính TRƯỚC khi thêm cột mới để chỉ gồm 3 cột điểm gốc
number_cols = df.select_dtypes(include="number").columns.tolist()
CAC_DIEM = number_cols
print("Các cột điểm:", CAC_DIEM)

Đã đọc: ../shared/students_cleaned.csv | kích thước: (1000, 8)
Các cột: ['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch', 'test_preparation_course', 'math_score', 'reading_score', 'writing_score']
Các cột điểm: ['math_score', 'reading_score', 'writing_score']


## 2.1 Kỹ thuật Indexing và Lọc (Indexing and Filtering)

#### Dùng `.loc[]`, `.iloc[]` để truy cập dữ liệu, rồi lọc bằng Boolean Indexing trên 2 cột.

### 2.1.1 Truy cập dữ liệu bằng `.loc[]` và `.iloc[]`

In [2]:
# ---------- .loc[]: truy cập theo NHÃN (tên dòng, tên cột) ----------
# Lấy 10 dòng đầu tiên (nhãn 0 đến 9; .loc bao gồm cả nhãn cuối nên 0:9 cho đúng 10 dòng) và 4 cột chỉ định theo tên
print("===== .loc[]: 10 dòng đầu, 4 cột chọn theo tên =====")
print(df.loc[0:9, ["gender", "math_score", "reading_score", "writing_score"]])

# ---------- .iloc[]: truy cập theo VỊ TRÍ (số thứ tự dòng, số thứ tự cột) ----------
# Lấy 10 dòng đầu (vị trí 0 đến 9; .iloc KHÔNG bao gồm vị trí cuối nên :10 cho đúng 10 dòng)
# và các cột ở vị trí 0, 5, 6, 7 (tương ứng gender, math_score, reading_score, writing_score)
print("\n===== .iloc[]: 10 dòng đầu, các cột ở vị trí 0, 5, 6, 7 =====")
print(df.iloc[:10, [0, 5, 6, 7]])

# Kiểm tra lại tên của các cột ở vị trí 0, 5, 6, 7 để chắc chắn chọn đúng
print("\nTên các cột ở vị trí 0, 5, 6, 7:", df.columns[[0, 5, 6, 7]].tolist())

===== .loc[]: 10 dòng đầu, 4 cột chọn theo tên =====
   gender  math_score  reading_score  writing_score
0  female          72             72             74
1  female          69             90             88
2  female          90             95             93
3    male          47             57             44
4    male          76             78             75
5  female          71             83             78
6  female          88             95             92
7    male          40             43             39
8    male          64             64             67
9  female          38             60             50

===== .iloc[]: 10 dòng đầu, các cột ở vị trí 0, 5, 6, 7 =====
   gender  math_score  reading_score  writing_score
0  female          72             72             74
1  female          69             90             88
2  female          90             95             93
3    male          47             57             44
4    male          76             78             75


**Nhận xét:**
- `.loc[0:9, [...]]` chọn theo **nhãn** nên lấy cả nhãn 9 (đủ 10 dòng), còn `.iloc[:10, [...]]` chọn theo **vị trí** nên dừng trước vị trí 10 (cũng đủ 10 dòng). Do chỉ số của bảng là 0, 1, 2, ... liên tục nên hai cách cho cùng kết quả.
- Dùng `.loc` khi biết **tên** dòng/cột (dễ đọc, không sai khi thứ tự cột thay đổi); dùng `.iloc` khi chỉ biết **vị trí**. Ở ví dụ trên, `.iloc` phải đối chiếu lại tên cột ở vị trí 0, 5, 6, 7 (gender, math_score, reading_score, writing_score) để chắc chắn chọn đúng.

### 2.1.2 Lọc dữ liệu theo điều kiện trên 2 cột (Boolean Indexing)

In [3]:
# ---------- Ví dụ 1: học sinh có điểm toán >= 80 VÀ điểm đọc >= 80 ----------
# Mỗi điều kiện phải đặt trong ngoặc tròn; dùng & cho "và", | cho "hoặc", ~ cho "phủ định"
dieu_kien_1 = (df["math_score"] >= 80) & (df["reading_score"] >= 80)
# Lấy các dòng thoả điều kiện bằng cách đưa dãy True/False vào df[...]
nhom_gioi = df[dieu_kien_1]
# In số học sinh thoả điều kiện và tỷ lệ phần trăm so với toàn bộ dữ liệu
print("Điểm toán >= 80 và điểm đọc >= 80:", len(nhom_gioi), "học sinh (", round(len(nhom_gioi) / len(df) * 100, 1), "% )")
# In 5 dòng đầu của nhóm
print(nhom_gioi.head())

# ---------- Ví dụ 2: học sinh ăn trưa miễn/giảm (free/reduced) VÀ điểm toán < 50 ----------
dieu_kien_2 = (df["lunch"] == "free/reduced") & (df["math_score"] < 50)
nhom_yeu = df[dieu_kien_2]
# In số học sinh thoả điều kiện
print("\nlunch = free/reduced và điểm toán < 50:", len(nhom_yeu), "học sinh")
# So sánh với toàn bộ học sinh có điểm toán < 50: trong đó bao nhiêu thuộc nhóm free/reduced (value_counts đếm số lần xuất hiện)
print("\nCác học sinh có điểm toán < 50 chia theo lunch:")
print(df[df["math_score"] < 50]["lunch"].value_counts())

Điểm toán >= 80 và điểm đọc >= 80: 143 học sinh ( 14.3 % )
    gender race_ethnicity parental_level_of_education     lunch  \
2   female        group B             master's degree  standard   
6   female        group B                some college  standard   
16    male        group C                 high school  standard   
34    male        group E                some college  standard   
35    male        group E          associate's degree  standard   

   test_preparation_course  math_score  reading_score  writing_score  
2                     none          90             95             93  
6                completed          88             95             92  
16                    none          88             89             86  
34                    none          97             87             82  
35               completed          81             81             79  

lunch = free/reduced và điểm toán < 50: 96 học sinh

Các học sinh có điểm toán < 50 chia theo lunch:
lunch
free

**Nhận xét:**
- Có **143 học sinh (14,3%)** đạt từ 80 điểm trở lên ở cả toán và đọc.
- Có **96 học sinh** vừa ăn trưa miễn/giảm (`free/reduced`) vừa dưới 50 điểm toán. Trong tổng số 135 học sinh dưới 50 điểm toán, nhóm `free/reduced` chiếm khoảng **71%**, trong khi nhóm này chỉ chiếm 35,5% toàn bộ mẫu (355/1.000). Như vậy học sinh `free/reduced` xuất hiện nhiều bất thường trong nhóm điểm toán thấp.

## 2.2 Kỹ thuật Feature Engineering (Tạo thuộc tính mới)

#### Tạo ít nhất ba cột mới từ các cột có sẵn và chèn vào vị trí hợp lý bằng `.insert()`.

Tạo 4 cột mới như sau:

| Cột mới | Tạo từ | Ý nghĩa |
|---|---|---|
| `tong_diem` | 3 cột điểm | Tổng điểm 3 môn (tối đa 300) |
| `diem_trung_binh` | 3 cột điểm | Điểm trung bình 3 môn |
| `xep_loai` | `diem_trung_binh` | Xếp loại: Yếu (< 50), Trung bình (50 đến < 65), Khá (65 đến < 80), Giỏi (≥ 80) |
| `ket_qua` | 3 cột điểm | "Đạt" nếu cả 3 môn đều ≥ 50, ngược lại "Không đạt" |

In [4]:
# .insert(vị_trí, tên_cột, giá_trị) chèn cột mới vào đúng vị trí; vị trí lấy bằng df.columns.get_loc(tên cột)
# Vị trí ngay sau cột điểm cuối cùng (writing_score)
vi_tri = df.columns.get_loc("writing_score") + 1

# ---------- Cột 1: tong_diem, chèn ngay sau writing_score ----------
# sum(axis=1) cộng theo HÀNG, tức cộng 3 điểm của từng học sinh
df.insert(vi_tri, "tong_diem", df[CAC_DIEM].sum(axis=1))

# ---------- Cột 2: diem_trung_binh, chèn ngay sau tong_diem ----------
# mean(axis=1) lấy trung bình theo HÀNG; round(2) làm tròn 2 chữ số
df.insert(vi_tri + 1, "diem_trung_binh", df[CAC_DIEM].mean(axis=1).round(2))

# ---------- Cột 3: xep_loai, chèn ngay sau diem_trung_binh ----------
# pd.cut chia một cột số thành các khoảng; bins là các mốc chia, labels là tên của từng khoảng
# right=False nghĩa là khoảng đóng bên trái, mở bên phải: [0; 50), [50; 65), [65; 80), [80; 101)
df.insert(vi_tri + 2, "xep_loai",
          pd.cut(df["diem_trung_binh"], bins=[0, 50, 65, 80, 101], labels=["Yếu", "Trung bình", "Khá", "Giỏi"], right=False))

# ---------- Cột 4: ket_qua, chèn ở cuối bảng ----------
# (df[CAC_DIEM] >= 50).all(axis=1) là True khi CẢ 3 môn đều từ 50 điểm trở lên
# np.where(điều kiện, giá trị nếu đúng, giá trị nếu sai)
df.insert(len(df.columns), "ket_qua", np.where((df[CAC_DIEM] >= 50).all(axis=1), "Đạt", "Không đạt"))

# In các cột hiện có để thấy vị trí các cột mới
print("Các cột sau khi thêm:", df.columns.tolist())
# In 5 dòng đầu của các cột liên quan
print(df[CAC_DIEM + ["tong_diem", "diem_trung_binh", "xep_loai", "ket_qua"]].head())

# Đếm số lượng ở từng nhóm của hai cột phân loại mới để kiểm tra việc phân nhóm có hợp lý không
for cot in ["xep_loai", "ket_qua"]:
    print("\n--- " + cot + " ---")
    print(df[cot].value_counts().sort_index())

Các cột sau khi thêm: ['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch', 'test_preparation_course', 'math_score', 'reading_score', 'writing_score', 'tong_diem', 'diem_trung_binh', 'xep_loai', 'ket_qua']
   math_score  reading_score  writing_score  tong_diem  diem_trung_binh  \
0          72             72             74        218            72.67   
1          69             90             88        247            82.33   
2          90             95             93        278            92.67   
3          47             57             44        148            49.33   
4          76             78             75        229            76.33   

  xep_loai    ket_qua  
0      Khá        Đạt  
1     Giỏi        Đạt  
2     Giỏi        Đạt  
3      Yếu  Không đạt  
4      Khá        Đạt  

--- xep_loai ---
xep_loai
Yếu           103
Trung bình    296
Khá           403
Giỏi          198
Name: count, dtype: int64

--- ket_qua ---
ket_qua
Không đạt    188
Đạt          812


**Nhận xét:**
- Các cột mới nằm đúng vị trí mong muốn: `tong_diem`, `diem_trung_binh`, `xep_loai` nằm liền sau `writing_score`, còn `ket_qua` ở cuối bảng.
- Phân bố xếp loại: Khá nhiều nhất (403), sau đó Trung bình (296), Giỏi (198) và Yếu (103).
- Có **812 học sinh Đạt (81,2%)** và 188 học sinh Không đạt (18,8%), tức cứ khoảng 5 học sinh thì có 1 em dưới 50 điểm ở ít nhất một môn.

## 2.3 Phân tích Thống kê và Nhóm (Statistical Analysis and Grouping)

#### Tính ma trận tương quan, thống kê theo nhóm và phân tích tác động của một biến lên biến khác.

### 2.3.1 Ma trận tương quan (`.corr()`)

In [5]:
# .corr() chỉ tính được trên cột số, nên mã hoá 3 cột chữ có 2 giá trị thành 0/1 trước
# chuan_bi_num: 1 nếu đã hoàn thành khoá luyện thi (completed), 0 nếu chưa (none)
df["chuan_bi_num"] = (df["test_preparation_course"] == "completed").astype(int)
# lunch_num: 1 nếu ăn trưa tiêu chuẩn (standard), 0 nếu miễn/giảm (free/reduced)
df["lunch_num"] = (df["lunch"] == "standard").astype(int)
# nam_num: 1 nếu là nam, 0 nếu là nữ
df["nam_num"] = (df["gender"] == "male").astype(int)

# Danh sách các biến đưa vào ma trận tương quan
CAC_BIEN = CAC_DIEM + ["chuan_bi_num", "lunch_num", "nam_num"]

# Tính ma trận tương quan Pearson giữa tất cả các biến
ma_tran = df[CAC_BIEN].corr()
# In ma trận, làm tròn 3 chữ số thập phân
print(ma_tran.round(3))

# Chỉ lấy tương quan giữa 3 biến mã hoá (hàng) và 3 điểm số (cột) để thấy yếu tố nào liên quan đến điểm
print("\nTương quan giữa các yếu tố và điểm số:")
print(ma_tran.loc[["chuan_bi_num", "lunch_num", "nam_num"], CAC_DIEM].round(3))

               math_score  reading_score  writing_score  chuan_bi_num  \
math_score          1.000          0.818          0.803         0.178   
reading_score       0.818          1.000          0.955         0.242   
writing_score       0.803          0.955          1.000         0.313   
chuan_bi_num        0.178          0.242          0.313         1.000   
lunch_num           0.351          0.230          0.246        -0.017   
nam_num             0.168         -0.244         -0.301         0.006   

               lunch_num  nam_num  
math_score         0.351    0.168  
reading_score      0.230   -0.244  
writing_score      0.246   -0.301  
chuan_bi_num      -0.017    0.006  
lunch_num          1.000    0.021  
nam_num            0.021    1.000  

Tương quan giữa các yếu tố và điểm số:
              math_score  reading_score  writing_score
chuan_bi_num       0.178          0.242          0.313
lunch_num          0.351          0.230          0.246
nam_num            0.168       

**Nhận xét:**
- Ba môn tương quan dương mạnh với nhau: **reading và writing 0,955** (rất mạnh), math với reading 0,818 và math với writing 0,803. Học sinh giỏi một môn thường giỏi các môn còn lại, đặc biệt là hai môn ngôn ngữ.
- `lunch_num` tương quan mạnh nhất với **toán** (0,351); `chuan_bi_num` (hoàn thành khoá luyện thi) tương quan mạnh nhất với **writing** (0,313).
- `nam_num` có dấu khác nhau giữa các môn: **dương với toán (+0,168)** nhưng **âm với reading (−0,244) và writing (−0,301)**, nghĩa là nam nhỉnh hơn ở toán còn nữ nhỉnh hơn ở đọc và viết.
- Các hệ số của biến 0/1 chỉ ở mức yếu đến trung bình, và tương quan không đồng nghĩa với nhân quả.

### 2.3.2 Nhóm theo cột cụ thể và tính thống kê mô tả (trung bình, độ lệch chuẩn, min, max)

In [6]:
# ---------- Nhóm theo gender ----------
# groupby("gender") chia bảng thành 2 nhóm (female/male); .agg() tính nhiều thống kê cùng lúc cho từng cột điểm
print("===== Thống kê điểm theo gender =====")
print(df.groupby("gender")[CAC_DIEM].agg(["mean", "std", "min", "max"]).round(2))

# ---------- Nhóm theo test_preparation_course ----------
print("\n===== Thống kê điểm theo test_preparation_course =====")
print(df.groupby("test_preparation_course")[CAC_DIEM].agg(["mean", "std", "min", "max"]).round(2))

# ---------- Nhóm theo race_ethnicity: chỉ xét diem_trung_binh, thêm số học sinh (count) ----------
print("\n===== diem_trung_binh theo race_ethnicity =====")
print(df.groupby("race_ethnicity")["diem_trung_binh"].agg(["count", "mean", "std", "min", "max"]).round(2))

# ---------- Nhóm theo parental_level_of_education, sắp xếp theo điểm trung bình giảm dần ----------
print("\n===== diem_trung_binh theo parental_level_of_education =====")
print(df.groupby("parental_level_of_education")["diem_trung_binh"].agg(["count", "mean", "std", "min", "max"]).round(2).sort_values("mean", ascending=False))

===== Thống kê điểm theo gender =====
       math_score                 reading_score                 writing_score  \
             mean    std min  max          mean    std min  max          mean   
gender                                                                          
female      63.63  15.49   0  100         72.61  14.38  17  100         72.47   
male        68.73  14.36  27  100         65.47  13.93  23  100         63.31   

                        
          std min  max  
gender                  
female  14.84  10  100  
male    14.11  15  100  

===== Thống kê điểm theo test_preparation_course =====
                        math_score                 reading_score             \
                              mean    std min  max          mean    std min   
test_preparation_course                                                       
completed                    69.70  14.44  23  100         73.89  13.64  37   
none                         64.08  15.19   0  100         

**Nhận xét:**
- **Theo `gender`:** nữ cao hơn ở reading (72,61 so với 65,47) và writing (72,47 so với 63,31), nam cao hơn ở math (68,73 so với 63,63). Độ lệch chuẩn của hai giới gần như bằng nhau (khoảng 14 đến 15 điểm).
- **Theo `test_preparation_course`:** nhóm đã hoàn thành khoá luyện thi có điểm trung bình cao hơn ở cả 3 môn, chênh nhiều nhất ở writing (74,42 so với 64,50). Điểm thấp nhất (math = 0, writing = 10) đều thuộc nhóm chưa luyện thi.
- **Theo `race_ethnicity`:** điểm trung bình tăng dần từ group A (62,99) đến group E (72,75), chênh khoảng 10 điểm. Tuy nhiên độ lệch chuẩn trong mỗi nhóm khoảng 13 đến 15 điểm nên các nhóm chồng lấn nhiều.
- **Theo `parental_level_of_education`:** điểm giảm đều theo trình độ học vấn của phụ huynh, từ 73,60 (master's degree) xuống 63,10 (high school). Nhóm master's degree chỉ có 59 học sinh nên cần thận trọng khi so sánh.

### 2.3.3 Phân tích tác động của một biến lên biến khác bằng Grouping và Aggregation

Biến tác động: `test_preparation_course`, `lunch`, `parental_level_of_education`, `gender`. Biến chịu tác động: điểm số (`diem_trung_binh` và từng môn).

In [7]:
# ---------- Tác động của khoá luyện thi (test_preparation_course) lên điểm số ----------
# Điểm trung bình từng môn của hai nhóm: đã hoàn thành (completed) và chưa (none)
tb_luyen_thi = df.groupby("test_preparation_course")[CAC_DIEM + ["diem_trung_binh"]].mean()
print("Điểm trung bình theo test_preparation_course:")
print(tb_luyen_thi.round(2))
# Chênh lệch giữa hai nhóm (completed trừ none) cho từng môn
print("\nChênh lệch (completed - none):")
print((tb_luyen_thi.loc["completed"] - tb_luyen_thi.loc["none"]).round(2))

# ---------- Tác động của loại bữa ăn trưa (lunch) lên điểm số ----------
tb_lunch = df.groupby("lunch")[CAC_DIEM + ["diem_trung_binh"]].mean()
print("\nĐiểm trung bình theo lunch:")
print(tb_lunch.round(2))
print("\nChênh lệch (standard - free/reduced):")
print((tb_lunch.loc["standard"] - tb_lunch.loc["free/reduced"]).round(2))

# ---------- Tác động đồng thời của gender và test_preparation_course lên điểm trung bình ----------
# pivot_table: hàng = gender, cột = test_preparation_course, giá trị = điểm trung bình
print("\n===== diem_trung_binh theo gender và test_preparation_course =====")
print(df.pivot_table(index="gender", columns="test_preparation_course", values="diem_trung_binh", aggfunc="mean").round(2))

# ---------- Tác động của gender lên từng môn ----------
# Chênh lệch trung bình giữa nam và nữ ở từng môn (male trừ female)
tb_gioi = df.groupby("gender")[CAC_DIEM].mean()
print("\nChênh lệch (male - female):")
print((tb_gioi.loc["male"] - tb_gioi.loc["female"]).round(2))

# ---------- Tỷ lệ Đạt / Không đạt theo lunch và theo test_preparation_course ----------
# crosstab đếm số học sinh theo hai cột; normalize="index" đổi thành tỷ lệ theo từng hàng, nhân 100 để ra phần trăm
print("\n===== Tỷ lệ (%) Đạt / Không đạt theo lunch =====")
print((pd.crosstab(df["lunch"], df["ket_qua"], normalize="index") * 100).round(1))
print("\n===== Tỷ lệ (%) Đạt / Không đạt theo test_preparation_course =====")
print((pd.crosstab(df["test_preparation_course"], df["ket_qua"], normalize="index") * 100).round(1))

Điểm trung bình theo test_preparation_course:
                         math_score  reading_score  writing_score  \
test_preparation_course                                             
completed                     69.70          73.89          74.42   
none                          64.08          66.53          64.50   

                         diem_trung_binh  
test_preparation_course                   
completed                          72.67  
none                               65.04  

Chênh lệch (completed - none):
math_score         5.62
reading_score      7.36
writing_score      9.91
diem_trung_binh    7.63
dtype: float64

Điểm trung bình theo lunch:
              math_score  reading_score  writing_score  diem_trung_binh
lunch                                                                  
free/reduced       58.92          64.65          63.02            62.20
standard           70.03          71.65          70.82            70.84

Chênh lệch (standard - free/reduced):
math_s

**Nhận xét:**
1. **Khoá luyện thi:** học sinh đã hoàn thành có điểm trung bình cao hơn **7,63 điểm** (5,62 ở toán, 7,36 ở đọc, 9,91 ở viết). Tỷ lệ Đạt là **89,9%** so với 76,3% ở nhóm chưa luyện thi.
2. **Loại bữa ăn trưa:** nhóm `standard` cao hơn nhóm `free/reduced` trung bình **8,64 điểm**, chênh nhiều nhất ở toán (11,11 điểm). Tỷ lệ Đạt là **88,1%** so với 68,7%, và tỷ lệ Không đạt của nhóm `free/reduced` (31,3%) gấp khoảng 2,6 lần nhóm `standard` (11,9%).
3. **Giới tính kết hợp khoá luyện thi:** khoá luyện thi làm tăng điểm khoảng 7,6 đến 7,7 điểm ở cả nữ (66,88 lên 74,46) và nam (63,04 lên 70,78), nên tác động của luyện thi gần như giống nhau ở hai giới.
4. **Giới tính theo từng môn:** nam hơn nữ 5,10 điểm ở toán nhưng kém 7,14 điểm ở đọc và 9,16 điểm ở viết.
5. Đây là dữ liệu quan sát. Loại bữa ăn trưa có thể phản ánh điều kiện kinh tế gia đình chứ bản thân bữa ăn chưa chắc là nguyên nhân làm điểm thay đổi.

## 2.4 Chuyển đổi Dữ liệu (Transformation)

#### Chuẩn hoá điểm toán trong phạm vi từng nhóm dữ liệu bằng `groupby().transform()`.

Chuẩn hoá cột `math_score` (`TARGET`) **trong phạm vi từng nhóm** `race_ethnicity` (`GROUP_COL`) bằng `groupby().transform()`:

- **Min-Max:** $x' = \dfrac{x - \min}{\max - \min}$, đưa mọi giá trị về đoạn [0, 1].
- **Z-score:** $z = \dfrac{x - \bar{x}}{s}$, đưa trung bình về 0 và độ lệch chuẩn về 1.

`transform()` trả về một Series có **cùng độ dài** với bảng gốc, nên gán được thẳng thành cột mới.

In [8]:
# ---------- Chuẩn hoá Min-Max theo từng nhóm race_ethnicity ----------
def min_max(s):
    """Đưa các giá trị của một nhóm về đoạn [0, 1]: (x - min) / (max - min)."""
    # Khoảng cách giữa giá trị lớn nhất và nhỏ nhất của nhóm
    khoang = s.max() - s.min()
    # Nếu cả nhóm có cùng một giá trị (khoảng = 0) thì trả về 0 để tránh chia cho 0
    return (s - s.min()) / khoang if khoang != 0 else s * 0

# Áp dụng hàm cho cột math_score trong từng nhóm, kết quả gán vào cột mới math_minmax
df["math_minmax"] = df.groupby(GROUP_COL)[TARGET].transform(min_max)

# ---------- Chuẩn hoá Z-score theo từng nhóm race_ethnicity ----------
# Với mỗi nhóm: trừ trung bình của nhóm rồi chia cho độ lệch chuẩn của nhóm
df["math_zscore"] = df.groupby(GROUP_COL)[TARGET].transform(lambda s: (s - s.mean()) / s.std())

# In 8 dòng đầu để so sánh giá trị gốc với hai giá trị đã chuẩn hoá
print(df[[GROUP_COL, TARGET, "math_minmax", "math_zscore"]].head(8).round(4))

# ---------- Kiểm tra kết quả ----------
# Min-Max: mỗi nhóm phải có min = 0 và max = 1
print("\n===== Kiểm tra Min-Max theo nhóm (min phải = 0, max phải = 1) =====")
print(df.groupby(GROUP_COL)["math_minmax"].agg(["min", "max", "mean"]).round(4))

# Z-score: mỗi nhóm phải có trung bình xấp xỉ 0 và độ lệch chuẩn = 1
print("\n===== Kiểm tra Z-score theo nhóm (mean phải ~ 0, std phải = 1) =====")
print(df.groupby(GROUP_COL)["math_zscore"].agg(["mean", "std"]).round(4))

# ---------- So sánh với chuẩn hoá toàn cục (không chia nhóm) ----------
# Nếu chuẩn hoá chung cho cả bảng thì điểm trung bình của các nhóm vẫn khác nhau, chuẩn hoá theo nhóm đưa mọi nhóm về cùng thang
df["math_minmax_chung"] = (df[TARGET] - df[TARGET].min()) / (df[TARGET].max() - df[TARGET].min())
print("\n===== Trung bình sau chuẩn hoá Min-Max: chung toàn bộ và theo nhóm =====")
print(df.groupby(GROUP_COL)[["math_minmax_chung", "math_minmax"]].mean().round(3))

  race_ethnicity  math_score  math_minmax  math_zscore
0        group B          72       0.7191       0.5526
1        group C          69       0.7041       0.3054
2        group B          90       0.9213       1.7163
3        group A          47       0.2639      -1.0073
4        group C          76       0.7755       0.7767
5        group B          71       0.7079       0.4879
6        group B          88       0.8989       1.5870
7        group B          40       0.3596      -1.5162

===== Kiểm tra Min-Max theo nhóm (min phải = 0, max phải = 1) =====
                min  max    mean
race_ethnicity                  
group A         0.0  1.0  0.4671
group B         0.0  1.0  0.6231
group C         0.0  1.0  0.6578
group D         0.0  1.0  0.5590
group E         0.0  1.0  0.6260

===== Kiểm tra Z-score theo nhóm (mean phải ~ 0, std phải = 1) =====
                mean  std
race_ethnicity           
group A          0.0  1.0
group B         -0.0  1.0
group C          0.0  1.0
group

**Nhận xét:**
- Kiểm tra cho thấy chuẩn hoá đúng: mỗi nhóm `race_ethnicity` có Min-Max với min = 0, max = 1 và Z-score với trung bình = 0, độ lệch chuẩn = 1.
- Chuẩn hoá theo nhóm cho biết **vị trí tương đối của học sinh trong nhóm của mình**. Ví dụ học sinh ở dòng 3 (group A, 47 điểm toán) có Z-score −1,01, tức thấp hơn trung bình nhóm A khoảng 1 độ lệch chuẩn; học sinh ở dòng 2 (group B, 90 điểm) có Z-score +1,72, tức cao hơn trung bình nhóm B rõ rệt.
- So với chuẩn hoá chung cho toàn bộ dữ liệu, điểm trung bình Min-Max của các nhóm thay đổi (ví dụ group A từ 0,616 xuống 0,467, group E từ 0,738 xuống 0,626) vì mỗi nhóm được đưa về thang riêng của nó, nên không còn phản ánh việc nhóm này điểm cao hơn nhóm kia.
- Min-Max phụ thuộc vào điểm thấp nhất và cao nhất của nhóm nên **nhạy với giá trị cực đoan** (ví dụ có học sinh được 0 điểm toán, làm giá trị min của nhóm chứa em đó bằng 0). Z-score ít bị ảnh hưởng hơn bởi một giá trị đơn lẻ.

In [9]:
# Lưu bảng dữ liệu đã thêm cột mới và chuẩn hoá để dùng cho Phần III (trực quan hoá) và Phần IV (báo cáo)
if os.path.exists("../shared/"):
    df.to_csv("../shared/students_part2.csv", index=False)
    print("Đã lưu vào ../shared/students_part2.csv")
else:
    df.to_csv("students_part2.csv", index=False)
    print("Đã lưu vào thư mục hiện tại (students_part2.csv)")
print("Kích thước cuối cùng:", df.shape)
print("Các cột:", df.columns.tolist())

Đã lưu vào ../shared/students_part2.csv
Kích thước cuối cùng: (1000, 18)
Các cột: ['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch', 'test_preparation_course', 'math_score', 'reading_score', 'writing_score', 'tong_diem', 'diem_trung_binh', 'xep_loai', 'ket_qua', 'chuan_bi_num', 'lunch_num', 'nam_num', 'math_minmax', 'math_zscore', 'math_minmax_chung']


## Kết luận Phần II

1. **Indexing và lọc:** `.loc` và `.iloc` truy cập đúng 10 dòng đầu; Boolean indexing cho thấy 143 học sinh (14,3%) đạt từ 80 điểm toán và đọc, và học sinh `free/reduced` chiếm khoảng 71% số em dưới 50 điểm toán.
2. **Feature Engineering:** thêm 4 cột `tong_diem`, `diem_trung_binh`, `xep_loai`, `ket_qua` bằng `.insert()`; có 81,2% học sinh Đạt.
3. **Thống kê và nhóm:** reading và writing tương quan rất mạnh (0,955); hoàn thành khoá luyện thi làm điểm trung bình cao hơn khoảng 7,6 điểm, ăn trưa `standard` cao hơn khoảng 8,6 điểm; nam hơn nữ ở toán, nữ hơn nam ở đọc và viết; điểm tăng theo trình độ học vấn của phụ huynh.
4. **Chuyển đổi:** chuẩn hoá `math_score` theo từng nhóm `race_ethnicity` bằng `groupby().transform()` (Min-Max và Z-score), kiểm tra đúng min/max và mean/std.

Dữ liệu sau Phần II được lưu thành `students_part2.csv` để dùng cho Phần III (ví dụ biểu đồ tròn của `xep_loai`) và Phần IV.